# AI-Driven Dynamic Pricing System for Delivery Services: A Deep Learning Approach to Revenue Optimization
### Strategic Modeling of Price Elasticity and Contextual Demand

*by ABDULRAHMAN JABER AGEELI, June, 2026* 

**University:** Umm Al-Qura University  
**Course:** Deep Learning Project

## 1. Introduction
A primary challenge in modern logistics is identifying the 'Optimal Price'. Conventional pricing models often rely on fixed margins, which typically fail to capture the complex behavior of human acceptance. This project develops an **Applied AI System** that utilizes a **Deep Neural Network (DNN)** to model price elasticity and maximize **Expected Revenue**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve, confusion_matrix
from sklearn.calibration import calibration_curve

# Global Configurations
pd.set_option('display.max_columns', None)
rng = np.random.default_rng(42) # Statistical consistency

## 2. Methodology & Simulation
We use a stochastic environment with a dynamic **Traffic Factor** to simulate realistic delivery durations and distance-based base prices ($p_{base}$).

In [ ]:
def apply_features(data):
    data = data.copy()
    eps = 1e-6
    data["price_dist_ratio"] = data["price"] / (data["distance"] + eps)
    data["price_base_ratio"] = data["price"] / data["p_base"]
    return data

def simulate_market_system(data, n_prices=10):
    data_rep = pd.concat([data] * n_prices, ignore_index=True)
    n = len(data_rep)
    alpha = (0.2 + 0.05 * data_rep["distance_km"] - 0.08 * data_rep["is_peak"])
    delta = rng.uniform(-0.4 * data_rep["p_base"], 0.5 * data_rep["p_base"], n)
    price = np.maximum(2.0, data_rep["p_base"] + delta)
    z = -alpha * (price - data_rep["p_base"]) + rng.normal(0, 0.5, n)
    target = (rng.random(n) < (1 / (1 + np.exp(-z)))).astype(int)
    
    sim_df = pd.DataFrame({
        "distance": data_rep["distance_km"], "is_peak": data_rep["is_peak"],
        "price": price, "p_base": data_rep["p_base"], "target": target
    })
    return apply_features(sim_df)

## 3. Deep Learning Architecture
A DNN with **Batch Normalization** and **Dropout** is used to predict the probability of acceptance.

In [ ]:
def build_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_shape,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

## 4. Limitations and Future Work
- The model relies on simulated demand; real-world validation is needed.
- External factors like competition or weather are not currently included.